In [5]:
# Middlreware

In [6]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

python-dotenv could not parse statement starting at line 8


In [7]:
# summarization middle ware 

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [9]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [10]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='77301676-d14f-40a1-926e-443c2a5a8a91'), AIMessage(content='<think>\nOkay, the user is asking "What is 2+2?" That seems straightforward. Let me think about how to approach this.\n\nFirst, I know that 2+2 is a basic arithmetic problem. The answer is 4, right? But maybe I should verify that. Let me count: two plus two... yes, that\'s four. I can also visualize it with objects. If I have two apples and add two more, I have four apples total. \n\nWait, is there any context where 2+2 might not equal 4? In standard arithmetic, yes, it\'s 4. But in some non-standard systems or different bases? For example, in base 3, 2+2 would be 11 (which is 4 in decimal). But the question doesn\'t specify a base, so it\'s safe to assume base 10. Also, in modular arithmetic, like modulo 3, 2+2 would be 1. But again, unless specified, the default is regular integer addition.\n\nAnother angle: sometimes 

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [12]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~166 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='ca7bc4c5-5c07-49f5-83ef-2ca5f639835c'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The city here is Paris. I need to make sure the function is called correctly. The parameters should be {"city": "Paris"}. Since the function is supposed to return a long response, I should expect that once I get the data, I\'ll need to present it in a detailed way. But first, just need to make the tool call with the right arguments. Let me confirm the JSON structure. The tool_call should have the name as "search_hotels" and the arguments as {"city": "Paris"}. Yep, that\'s all. No other parameters needed. Alright, time to format that into the required XML tags.\n', 'tool_calls': [{'id': '5jjh64zsk', 'function': {'ar

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~68 tokens (0.0531%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a72607af-7df6-4a0b-aa2d-ea7aefb1b028'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. Since the user specified Paris, I need to call this function with the city set to Paris. I should make sure the parameters are correctly formatted as JSON within the tool_call tags. No other parameters are needed, just the city. Alright, that should do it.\n", 'tool_calls': [{'id': '4e24hqe96', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 110, 'prompt_tokens': 147, 'total_tokens': 257, 'completion_time': 0.16861916, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.006204818, 'prompt_tokens

In [14]:
# Human in loop middle ware
# pause agent execution for human approval , editing , or rejection of tool calls before they execute . Human in the loop is useful for the following :
# high - stakes operations requirigin human approval (eg: database writes,finnancial transactions)
# Copliance workflows where human oversight is mandatory.
# Long running  conversations where human feedback guides the agent

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver



In [16]:
def read_email_tool(email_id: str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID:{email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [33]:
agent = create_agent(
    model= "groq:qwen/qwen3-32b",
    tools = [read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]   
                },
                "read_email_tool":False,
            }
        )
    ]

)

In [34]:
config = {"configurable":{"thread_id":"test-approve"}}
# step1 : Request
result = agent.invoke(
    {
        "messages": [HumanMessage(content="Senad email to john@test.com with subject 'Hello' and body 'How are you?' ")]
    },
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="Senad email to john@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='da155972-2666-4d76-aece-7c727b39f4db'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. So I need to call the send_email_tool with those arguments. No need to use the read_email_tool here since the user isn't asking to read an email. Just construct the JSON object with the parameters and make sure the keys are correct. Double-check the function name and parameters. Yep, looks good. Let me format the tool_call accordingly.\n

In [36]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email with the subject "Hello" and body "How are you?" has been successfully sent to john@test.com. Let me know if you need any further assistance!


In [37]:
result

{'messages': [HumanMessage(content="Senad email to john@test.com with subject 'Hello' and body 'How are you?' ", additional_kwargs={}, response_metadata={}, id='da155972-2666-4d76-aece-7c727b39f4db'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is 'Hello', and body is 'How are you?'. So I need to call the send_email_tool with those arguments. No need to use the read_email_tool here since the user isn't asking to read an email. Just construct the JSON object with the parameters and make sure the keys are correct. Double-check the function name and parameters. Yep, looks good. Let me format the tool_call accordingly.\n

In [38]:
# Reject

In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [40]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [41]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email to john@test.com could not be sent. Would you like to try again or check the details?


In [42]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='e0cbfdd0-d804-4d25-8ff3-664bb6b241b6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with these arguments. No need to use read_email_tool here since the user isn't asking to read an email. Just make sure the JSON is correctly formatted with the parameters. Double-check the spelling of the email address and the subject and body. Everything looks good. So the tool call should include all three parameters as specified.\n", 't

In [ ]:
# Editing

In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [44]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [45]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='827ce814-a310-4962-82c0-0a4aaad5699c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are provided: recipient is wrong@email.com, subject is 'Test', body is 'Hello'. So I need to call send_email_tool with these arguments. No issues here, all required parameters are present. Just format the JSON correctly.\n", 'tool_calls': [{'id': 'z4d5s5qcr', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 249, 'total_tokens

In [46]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email has been successfully sent to **correct@email.com** with the subject **"Corrected Subject"**. I adjusted the recipient and subject to ensure it reaches the intended address. Let me know if you need further assistance!
